# DiploDatos 2026
# Entregable 1: Analisis y Visualizacion
---
# Predicciones en el Espacio: cuantos satelites y desechos podremos tener?
---

**Grupo:**

**Integrantes:**

**Fecha de entrega:**

## Proposito de este notebook

Este primer entregable tiene como objetivo construir una primera comprension del problema y del dataset `satellites_202602.csv`. El proyecto completo busca analizar objetos orbitales y, en etapas posteriores, desarrollar modelos que permitan predecir si un objeto continua activo (`ACTIVE = Yes`) o no (`ACTIVE = No`), o estimar su probabilidad de quedar fuera de servicio a partir de caracteristicas historicas, orbitales y de mision.

En este notebook todavia no se espera construir un modelo ni dejar un dataset definitivo. La meta es observar, visualizar y formular hipotesis: como evoluciono la poblacion de objetos en orbita?, que proporcion corresponde a satelites y a desechos?, que paises concentran mayor cantidad de objetos?, como se distribuyen las variables orbitales?, que senales preliminares podrian estar relacionadas con el estado operativo?

## Ubicacion dentro del proyecto

La secuencia de trabajo se organiza en tres notebooks principales:

1. **Analisis y visualizacion**: exploracion panoramica, graficos descriptivos, primeras preguntas e hipotesis.
2. **Analisis exploratorio y curacion de datos**: revision profunda de tipos de datos, faltantes, duplicados, inconsistencias, outliers, imputaciones, codificacion de variables y construccion del dataset curado.
3. **Aprendizaje supervisado, no supervisado, o ambos**: entrenamiento, comparacion y evaluacion de modelos, interpretacion de resultados y discusion de utilidad para el problema.

Por eso, en este notebook las transformaciones deben ser livianas y estar al servicio de visualizar mejor. Toda decision fuerte de limpieza o preparacion para modelado debe quedar documentada como pendiente para el siguiente entregable.

## Objetivos del entregable

- Familiarizarse con las columnas del dataset y su significado en el contexto espacial.
- Dimensionar el volumen total de objetos orbitales y la distribucion entre `PAYLOAD` y `DEBRIS`.
- Analizar la variable objetivo futura `ACTIVE` desde una perspectiva descriptiva, sin entrenar modelos.
- Visualizar tendencias temporales de lanzamientos, acumulacion de objetos y cambios recientes.
- Comparar patrones por pais, tipo de objeto, estado operativo, proposito y variables orbitales.
- Detectar senales preliminares de calidad de datos, sin resolver todavia la curacion completa.
- Formular hipotesis concretas para validar en los proximos notebooks.

## Preguntas guia

- Cuantos objetos orbitales contiene el dataset y como se distribuyen entre satelites/cargas utiles y desechos?
- Que proporcion de objetos figura como activa e inactiva?
- Como evoluciono la cantidad de lanzamientos por ano? Se observa un cambio desde 2019?
- Que paises u organizaciones concentran mayor cantidad de objetos orbitales?
- Que propositos de mision aparecen con mas frecuencia entre los objetos activos?
- Como se distribuyen variables orbitales como `PERIOD`, `INCLINATION`, `APOGEE` y `PERIGEE`?
- Hay diferencias visibles entre objetos activos e inactivos segun variables orbitales o historicas?
- Que valores faltantes o codigos especiales conviene revisar con mas detalle en la etapa de curacion?
- Que hipotesis podrian transformarse luego en variables predictivas o criterios de segmentacion?

## 1. Preparacion del ambiente

Cargamos las librerias necesarias para una primera exploracion. Se sugiere mantener el codigo simple y reproducible: cada grafico debe poder volver a generarse desde el CSV original.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("../data/raw/satellites_202602.csv")

## 2. Ingesta de datos

Leer el archivo original y realizar una primera inspeccion de dimensiones, columnas y registros. En esta etapa no se modifica el dataset fuente.

In [ ]:
data = pd.read_csv(DATA_PATH)

print(f"Filas: {data.shape[0]:,}")
print(f"Columnas: {data.shape[1]:,}")
data.head(10)

In [ ]:
data.info()

### Columnas relevantes para iniciar el analisis

Algunas variables que conviene mirar desde el comienzo:

- `OBJECT_TYPE`: tipo de objeto registrado, principalmente `PAYLOAD` o `DEBRIS`.
- `ACTIVE`: variable objetivo futura para clasificacion supervisada.
- `COUNTRY`: pais u organizacion responsable.
- `LAUNCH` y `LAUNCH_YEAR`: fecha y ano de lanzamiento.
- `DECAY`: fecha de reentrada o decaimiento orbital, cuando esta disponible.
- `PERIOD`, `INCLINATION`, `APOGEE`, `PERIGEE`: parametros orbitales.
- `RCS_SIZE` y `RCSVALUE`: tamano o seccion radar estimada.
- `PURPOSE`: proposito de mision, especialmente util para satelites activos.

In [ ]:
column_summary = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "non_null": data.notna().sum(),
    "nulls": data.isna().sum(),
    "null_pct": (data.isna().mean() * 100).round(2),
    "unique_values": data.nunique(dropna=True),
})

column_summary.sort_values("null_pct", ascending=False)

**Para interpretar:** los valores faltantes no siempre significan error. Por ejemplo, `DECAY` puede estar vacio porque el objeto sigue en orbita. En este notebook alcanza con detectar y comentar estos casos; la decision de imputar, descartar o recodificar corresponde al notebook de curacion.

## 3. Variables derivadas solo para visualizar

Creamos una copia de trabajo con conversiones minimas y algunas variables auxiliares. Estas variables ayudan a graficar, pero no deben considerarse todavia como el dataset final del proyecto.

In [ ]:
df = data.copy()

for col in ["LAUNCH", "DECAY"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Altitud media aproximada, util para visualizar la distribucion orbital.
df["MEAN_ALTITUDE"] = df[["APOGEE", "PERIGEE"]].mean(axis=1)

# Clasificacion orbital simplificada para analisis descriptivo inicial.
# Los limites son aproximados y deberan revisarse si se usan para modelado.
def classify_orbit(row):
    apogee = row["APOGEE"]
    perigee = row["PERIGEE"]
    if pd.isna(apogee) or pd.isna(perigee):
        return "Sin datos"
    mean_altitude = (apogee + perigee) / 2
    if mean_altitude < 2_000:
        return "LEO aprox."
    if 30_000 <= mean_altitude <= 40_000:
        return "GEO/GSO aprox."
    if mean_altitude < 30_000:
        return "MEO/HEO aprox."
    return "Orbita alta aprox."

df["ORBIT_CLASS_APPROX"] = df.apply(classify_orbit, axis=1)

df[["OBJECT_TYPE", "ACTIVE", "COUNTRY", "LAUNCH_YEAR", "MEAN_ALTITUDE", "ORBIT_CLASS_APPROX"]].head()

## 4. Panorama general del dataset

Comenzamos con conteos simples. Estos graficos deben ayudar a responder que contiene el dataset antes de hacer analisis mas finos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

object_counts = df["OBJECT_TYPE"].value_counts()
object_counts.plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].set_title("Objetos por tipo")
axes[0].set_xlabel("Tipo de objeto")
axes[0].set_ylabel("Cantidad")
axes[0].tick_params(axis="x", rotation=0)

active_counts = df["ACTIVE"].value_counts(dropna=False)
active_counts.plot(kind="bar", ax=axes[1], color=["#54A24B", "#E45756"])
axes[1].set_title("Objetos segun estado activo")
axes[1].set_xlabel("ACTIVE")
axes[1].set_ylabel("Cantidad")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(df["OBJECT_TYPE"], df["ACTIVE"], margins=True, normalize="index").round(3)

**Consignas de interpretacion**

- Describir la diferencia entre cantidad total de `DEBRIS` y `PAYLOAD`.
- Analizar si `ACTIVE` esta balanceada o desbalanceada.
- Explicar por que este desbalance puede ser importante para el futuro notebook de aprendizaje supervisado.

## 5. Evolucion temporal

El crecimiento temporal es central para el problema: permite observar la expansion de objetos orbitales, la aparicion de nuevas etapas de lanzamiento y el posible aumento de congestion espacial.

In [ ]:
launches_by_year = df.groupby("LAUNCH_YEAR").size()

fig, ax = plt.subplots(figsize=(12, 5))
launches_by_year.plot(ax=ax, color="#4C78A8")
ax.set_title("Objetos registrados por ano de lanzamiento")
ax.set_xlabel("Ano de lanzamiento")
ax.set_ylabel("Cantidad de objetos")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
launches_by_year_and_type = (
    df.pivot_table(index="LAUNCH_YEAR", columns="OBJECT_TYPE", values="NORAD_CAT_ID", aggfunc="count")
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(12, 5))
launches_by_year_and_type.plot(ax=ax)
ax.set_title("Evolucion por tipo de objeto")
ax.set_xlabel("Ano de lanzamiento")
ax.set_ylabel("Cantidad de objetos")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
launches_by_year.cumsum().plot(ax=ax, color="#72B7B2")
ax.set_title("Acumulado historico de objetos registrados")
ax.set_xlabel("Ano de lanzamiento")
ax.set_ylabel("Cantidad acumulada")
ax.grid(True, alpha=0.3)
plt.show()

**Consignas de interpretacion**

- Identificar periodos de crecimiento acelerado.
- Comparar la evolucion de `PAYLOAD` y `DEBRIS`.
- Discutir si el periodo reciente cambia la escala del problema.
- Formular una hipotesis sobre el efecto de megaconstelaciones o nuevos actores privados, si el patron observado lo justifica.

## 6. Paises, organizaciones y propositos

Estos graficos permiten observar concentracion de actividad orbital y posibles diferencias entre actores. No alcanza con contar: tambien conviene relacionar pais, tipo de objeto y estado activo.

In [ ]:
top_countries = df["COUNTRY"].value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
top_countries.plot(kind="barh", ax=ax, color="#4C78A8")
ax.set_title("Top 15 paises/organizaciones por cantidad de objetos")
ax.set_xlabel("Cantidad")
ax.set_ylabel("Pais u organizacion")
plt.show()

In [ ]:
top_country_names = df["COUNTRY"].value_counts().head(10).index
country_type_table = pd.crosstab(
    df.loc[df["COUNTRY"].isin(top_country_names), "COUNTRY"],
    df.loc[df["COUNTRY"].isin(top_country_names), "OBJECT_TYPE"],
)

country_type_table.sort_values(country_type_table.columns.tolist(), ascending=False).plot(
    kind="bar", stacked=True, figsize=(12, 5)
)
plt.title("Tipo de objeto en los 10 paises/organizaciones con mas registros")
plt.xlabel("Pais u organizacion")
plt.ylabel("Cantidad")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
active_payloads = df[(df["OBJECT_TYPE"] == "PAYLOAD") & (df["ACTIVE"] == "Yes")]

purpose_counts = active_payloads["PURPOSE"].fillna("Sin datos").value_counts().head(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
purpose_counts.plot(kind="barh", ax=ax, color="#F58518")
ax.set_title("Propositos mas frecuentes entre payloads activos")
ax.set_xlabel("Cantidad")
ax.set_ylabel("Proposito")
plt.show()

**Consignas de interpretacion**

- Senalar que actores concentran mas objetos y si esa concentracion cambia segun `PAYLOAD` o `DEBRIS`.
- Observar que propositos dominan entre los satelites activos.
- Proponer agrupaciones de `PURPOSE` que podrian ser utiles para el notebook de curacion.

## 7. Variables orbitales

Las variables orbitales podrian aportar informacion relevante para explicar el estado operativo o la permanencia de objetos en orbita. En este notebook las miramos de forma descriptiva; el tratamiento de outliers, escalado y transformacion queda para mas adelante.

In [ ]:
orbital_cols = ["PERIOD", "INCLINATION", "APOGEE", "PERIGEE", "MEAN_ALTITUDE"]
df[orbital_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.ravel(), ["PERIOD", "INCLINATION", "APOGEE", "PERIGEE"]):
    df[col].dropna().plot(kind="hist", bins=60, ax=ax, color="#4C78A8", alpha=0.8)
    ax.set_title(f"Distribucion de {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
scatter_source = df.dropna(subset=["APOGEE", "PERIGEE", "ACTIVE"])
sample_for_scatter = scatter_source.sample(
    n=min(10_000, scatter_source.shape[0]),
    random_state=42,
)

colors = sample_for_scatter["ACTIVE"].map({"Yes": "#54A24B", "No": "#E45756"}).fillna("#9D9D9D")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(sample_for_scatter["PERIGEE"], sample_for_scatter["APOGEE"], c=colors, alpha=0.35, s=10)
ax.set_title("Apogeo vs. perigeo segun estado activo")
ax.set_xlabel("Perigeo [km]")
ax.set_ylabel("Apogeo [km]")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
orbit_active_table = pd.crosstab(df["ORBIT_CLASS_APPROX"], df["ACTIVE"], normalize="index").round(3)
orbit_active_table

**Consignas de interpretacion**

- Identificar distribuciones concentradas, sesgadas o con valores extremos.
- Comparar visualmente si los objetos activos ocupan regiones orbitales distintas a los inactivos.
- Decidir que variables orbitales merecen una revision de outliers en el siguiente notebook.

## 8. Senales preliminares de calidad de datos

Esta seccion no reemplaza la curacion. Su funcion es dejar registradas observaciones que condicionan los analisis posteriores.

In [ ]:
missing_overview = (
    df.isna().mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

missing_overview[missing_overview["missing_pct"] > 0]

In [ ]:
for col in ["COMMENT", "RCS_SIZE", "PURPOSE", "DECAY", "SITE"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(10))

**Consignas de interpretacion**

- Elegir tres columnas con faltantes relevantes y explicar que podrian significar en este dominio.
- Separar posibles errores de datos de ausencias esperables por definicion del problema.
- Registrar decisiones pendientes para el notebook de analisis exploratorio y curacion.

## 9. Insights preliminares e hipotesis

Completar esta seccion con conclusiones breves, apoyadas en los graficos anteriores. Cada hallazgo debe incluir evidencia: grafico, tabla o conteo que lo respalde.

### Hallazgo 1

Describir una tendencia temporal relevante.

### Hallazgo 2

Describir una diferencia entre `PAYLOAD` y `DEBRIS`.

### Hallazgo 3

Describir un patron por pais, organizacion o proposito.

### Hallazgo 4

Describir una senal relacionada con variables orbitales.

### Hallazgo 5

Describir un problema o decision pendiente de calidad de datos.

## 10. Entregables esperados

Al finalizar este notebook, el grupo deberia poder entregar:

- Una descripcion breve del dataset y de las variables mas importantes para el objetivo del proyecto.
- Al menos cinco visualizaciones interpretadas, no solo generadas.
- Un comentario sobre la distribucion de `ACTIVE` y su impacto potencial en modelado.
- Un analisis temporal de lanzamientos y acumulacion de objetos.
- Una comparacion por `OBJECT_TYPE`, `COUNTRY`, `PURPOSE` y variables orbitales.
- Una lista de problemas de datos o decisiones pendientes para el proximo notebook.
- Tres a cinco hipotesis que puedan validarse mediante curacion, ingenieria de variables o modelado.

## 11. Puente hacia los proximos notebooks

Para el notebook de **analisis exploratorio y curacion de datos**, dejar identificadas las columnas que requieren revision: fechas, faltantes, valores extremos, categorias poco frecuentes, variables redundantes, codificacion de `PURPOSE` y posible definicion de variables derivadas.

Para el notebook de **aprendizaje supervisado, no supervisado, o ambos**, dejar planteadas las hipotesis predictivas o de segmentacion: por ejemplo, si `LAUNCH_YEAR`, `OBJECT_TYPE`, `COUNTRY`, `PURPOSE`, `RCS_SIZE` o los parametros orbitales parecen asociarse con `ACTIVE` o con grupos naturales de objetos.